<a href="https://colab.research.google.com/github/ragiokay/ML-2026-HW7/blob/main/ML2026_Spring_HW7_Model_Merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ML2026 Spring HW7 — Model Merging**

在這份作業中，將實作 **Model Merging** 。

我們會使用兩個 7B 模型：
- `augmxnt/shisa-gamma-7b-v1`（擅長日文）
- `WizardLMTeam/WizardMath-7B-V1.1`（擅長數學推理）

**❗重要規則：**

- **請勿使用上述兩個模型以外的任何其他模型。**
- 你可以自由使用任何套件（不限於 mergekit）或自己撰寫的演算法來進行 merging。
- **Merging 的嚴格定義：合併後的模型總參數量必須與單一 base model 相同。** 換言之，MoE（Mixture of Experts）、Ensemble、Stacking 等會使 inference 時可使用的參數量增加的做法，皆視為違規。

你的目標是：
1. 先觀察兩個 base model 各自的能力
2. 嘗試各種 merge 方法，把兩個模型合併
3. 在日文數學題上測試合併後的模型表現，並將結果上傳到 judgeboi

# **Section 0: Preparing**

## Mount your notebook to your Drive

In [7]:
!git pull origin main

fatal: not a git repository (or any of the parent directories): .git


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Activate GPU
Model Merging 與 Inference 都需要 GPU，請務必啟用。

### **MUST READ**:

Colab does **NOT** guarantee the GPU access for free user ([ref](https://research.google.com/colaboratory/faq.html#idle-timeouts)). It is possible you get a message saying "Cannot connect to GPU backend" which means there are no enough GPU resources for you now. When this happens, you may need to **wait for one (or more) day or login different Google account to do the homework**.

### Enable GPU

1. Click on "Runtime" (or "執行階段") in the header.
2. Click on "Change runtime type" (or "變更執行階段類型") in the drop menu.
3. Select "T4 GPU" and save. (You can select "A100 GPU" or "V100 GPU" if you have Colab Pro)

## Check the status of your GPU

In [9]:
!nvidia-smi

Thu May 21 01:35:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install Packages & Download Data

安裝 `mergekit`（模型合併工具）及下載日文數學題目檔案。

In [10]:
# Install mergekit
!pip install -q mergekit

# Install / upgrade gdown for Google Drive download
!pip install -U -q gdown

# Download the Japanese math question file
!gdown 1_9OAxDwQgz1jUnlhHlkk3lLCTpY1Ib8Q -O /content/jp_math_question.json
# Verify download
import json
with open("/content/jp_math_question.json", "r") as f:
    questions = json.load(f)
print(f"成功載入 {len(questions)} 題日文數學題目！")
print(f"範例題目: {questions[0]['question'][:80]}...")
print(f"範例答案: {questions[0]['answer_number']}")

Downloading...
From: https://drive.google.com/uc?id=1_9OAxDwQgz1jUnlhHlkk3lLCTpY1Ib8Q
To: /content/jp_math_question.json
100% 4.68k/4.68k [00:00<00:00, 21.4MB/s]
成功載入 20 題日文數學題目！
範例題目: 太郎は平成5年に生まれました。次郎は令和2年に生まれました。太郎は次郎より何歳年上ですか？...
範例答案: 27


# **Section 1: Explore Base Models — 觀察兩個 Base Model 的能力**
## 若時間不足可以先跳到Section 2

在合併之前，我們先分別載入兩個 base model，觀察它們各自的強項與弱項。

- **shisa-gamma-7b-v1**：以日文能力見長的模型
- **WizardMath-7B-V1.1**：以數學推理見長的模型

我們會各丟一個 **日文問題** 和一個 **數學問題** 給兩個模型，直覺感受它們的差異。

## 1.1 定義推論用的 helper function

建立通用的推論與評估函數，後續兩個模型都會用到。

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """通用推論函數：給定模型與 prompt，回傳生成結果"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output[len(prompt):].strip()

In [ ]:
import re

############################################################
# Prompt 模板
USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

# ⛔ 嚴禁修改 USER_PROMPT_TEMPLATE！
#
# 本作業聚焦於 Model Merging 技術本身，
# 而非 Prompt Engineering，因此為確保所有同學的
# 評測條件一致，無論任何情況皆請勿修改以下 prompt。
# 違反此規則將視為違規。
############################################################


def extract_number(text):
    """從模型輸出中提取數字答案"""
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

def evaluate_model(model, tokenizer, questions, model_name):
    """在日文數學題集上評估模型"""
    correct = 0
    results = []
    print(f"\n{'='*60}")
    print(f"📊 評估模型: {model_name}")
    print(f"{'='*60}\n")

    for i, item in enumerate(questions):
        prompt = USER_PROMPT_TEMPLATE.format(question=item['question'])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                repetition_penalty=1.1
            )
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer_part = full_output[len(prompt):].strip()
        predicted = extract_number(answer_part)
        expected = item["answer_number"]
        is_correct = predicted == expected
        if is_correct:
            correct += 1
        results.append({
            "id": i + 1, "correct": is_correct,
            "predicted": predicted, "expected": expected,
            "raw": answer_part,
        })
        mark = "✅" if is_correct else "❌"
        print(f"{mark} Q{i+1:02d} | 預測: {str(predicted):>6s} | 正解: {expected:>6s}")
        print(f"     | 題目: {item['question'][:60]}")
        print(f"     | raw: {answer_part[:120]}")
        print()

    print(f"{'='*50}")
    print(f"🎯 {model_name} 正確率: {correct}/{len(questions)} ({correct/len(questions)*100:.1f}%)")
    print(f"{'='*50}")
    return correct, results

## 1.2 測試 Base Model ①：shisa-gamma-7b-v1（日文模型）

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("載入 shisa-gamma-7b-v1 ...")
shisa_name = "augmxnt/shisa-gamma-7b-v1"
shisa_tokenizer = AutoTokenizer.from_pretrained(shisa_name)
shisa_model = AutoModelForCausalLM.from_pretrained(
    shisa_name, torch_dtype=torch.float16, device_map="auto"
)
print("shisa-gamma-7b-v1 載入完成！")

### 日文能力測試（shisa）

In [ ]:
jp_prompt = "日本の四季の特徴について、それぞれ簡潔に説明してください。"

print("=" * 60)
print("📝 日文 Prompt:", jp_prompt)
print("=" * 60)
print("\n🔵 【shisa-gamma-7b（日文模型）】的回答：")
print("-" * 40)
shisa_jp_ans = generate_response(shisa_model, shisa_tokenizer, jp_prompt)
print(shisa_jp_ans[:500])

### 數學能力測試（shisa）

In [ ]:
math_prompt = (
    "Solve the following math problem step by step.\n\n"
    "Problem: A store sells apples for $2 each and oranges for $3 each. "
    "If a customer buys 5 apples and 4 oranges, and pays with a $50 bill, "
    "how much change should the customer receive?\n\n"
    "Solution:"
)

print("=" * 60)
print("📝 Math Prompt:")
print(math_prompt)
print("=" * 60)
print("\n🔵 【shisa-gamma-7b（日文模型）】的回答：")
print("-" * 40)
shisa_math_ans = generate_response(shisa_model, shisa_tokenizer, math_prompt)
print(shisa_math_ans[:500])

print("\n" + "=" * 60)
print("💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28")
print("=" * 60)

### 在日文數學題集上評估（shisa）

In [ ]:
with open("/content/jp_math_question.json", "r") as f:
    questions = json.load(f)

shisa_correct, shisa_results = evaluate_model(
    shisa_model, shisa_tokenizer, questions, "shisa-gamma-7b（日文模型）"
)

### 釋放 shisa 模型記憶體

In [ ]:
import gc

del shisa_model, shisa_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("已釋放 shisa-gamma-7b 記憶體！")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

## 1.3 測試 Base Model ②：WizardMath-7B-V1.1（數學模型）

In [ ]:
print("載入 WizardMath-7B-V1.1 ...")
wizard_name = "WizardLMTeam/WizardMath-7B-V1.1"
wizard_tokenizer = AutoTokenizer.from_pretrained(wizard_name)
wizard_model = AutoModelForCausalLM.from_pretrained(
    wizard_name, torch_dtype=torch.float16, device_map="auto"
)
print("WizardMath-7B-V1.1 載入完成！")

### 日文能力測試（WizardMath）

In [ ]:
print("=" * 60)
print("📝 日文 Prompt:", jp_prompt)
print("=" * 60)
print("\n🟠 【WizardMath-7B（數學模型）】的回答：")
print("-" * 40)
wizard_jp_ans = generate_response(wizard_model, wizard_tokenizer, jp_prompt)
print(wizard_jp_ans[:500])

### 數學能力測試（WizardMath）

In [ ]:
print("=" * 60)
print("📝 Math Prompt:")
print(math_prompt)
print("=" * 60)
print("\n🟠 【WizardMath-7B（數學模型）】的回答：")
print("-" * 40)
wizard_math_ans = generate_response(wizard_model, wizard_tokenizer, math_prompt)
print(wizard_math_ans[:500])

print("\n" + "=" * 60)
print("💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28")
print("=" * 60)

### 在日文數學題集上評估（WizardMath）

In [ ]:
wizard_correct, wizard_results = evaluate_model(
    wizard_model, wizard_tokenizer, questions, "WizardMath-7B（數學模型）"
)

In [ ]:
del wizard_model, wizard_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("已釋放 WizardMath-7B 記憶體！")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

### 🔍 Base Model Baseline

記下兩個 base model 的正確率，之後合併模型時可以做比較：
- shisa-gamma-7b：擅長日文，但數學推理能力可能不足
- WizardMath-7B：擅長數學推理，但可能無法理解日文題意

理想的 merge 應該要能結合兩者的優勢！

# **Section 2: Model Merging — 合併模型**

## 2.1 撰寫 Merge 設定檔

以下是一個最簡單的 **50/50 Linear Merge**（線性插值合併）設定。

### ⚠️ 這是你的主要修改區域！

你可以嘗試不同的合併策略，例如：
- **linear**：最直覺的加權平均（調整 `weight` 比例）
- **slerp**：球面線性插值（調整 `t` 參數）
- **ties**：TIES-Merging（需設定 `density` 與 `weight`）
- **dare_ties**：DARE + TIES（需設定 `density`、`weight`）

請參考 [mergekit 文件](https://github.com/arcee-ai/mergekit?tab=readme-ov-file#merge-methods) 了解更多合併方法與參數。

In [ ]:
############################################################
# ⚠️ 【可修改區域 — TODO】
#
# 目前最佳：dare_ties + Mistral-7B-v0.1 as base + layer gradient
# 測試正確率：~70%
#
# 提示：每次修改後需重新執行 Section 2 & 3

config = """
models:
  - model: augmxnt/shisa-gamma-7b-v1
    parameters:
      density: 0.7
      weight:
        - filter: self_attn
          value: [0.8, 0.6, 0.4]
        - filter: mlp
          value: [0.8, 0.5, 0.3]
        - value: 0.7
  - model: WizardLMTeam/WizardMath-7B-V1.1
    parameters:
      density: 0.7
      weight:
        - filter: self_attn
          value: [0.2, 0.4, 0.6]
        - filter: mlp
          value: [0.2, 0.5, 0.9]
        - value: 0.3
merge_method: dare_ties
base_model: mistralai/Mistral-7B-v0.1
parameters:
  rescale: false
dtype: float16
""".strip()

############################################################

with open("merge_config.yml", "w") as f:
    f.write(config)
print("設定檔寫入完成！內容如下：")
print("=" * 40)
print(config)
print("=" * 40)


In [ ]:
# 備存 merge_config.yml 到 Google Drive
import shutil, os
drive_backup_dir = "/content/drive/MyDrive/ML2026/HW7"
os.makedirs(drive_backup_dir, exist_ok=True)
shutil.copy("merge_config.yml", os.path.join(drive_backup_dir, "merge_config.yml"))
print(f"✅ merge_config.yml 已備存至 {drive_backup_dir}/merge_config.yml")

## 2.2 執行合併

使用 mergekit 進行模型合併（GPU 模式 + lazy unpickle 逐層載入以節省記憶體）。

由於模型大小為 7B 偏大，這個步驟根據不同方法可能需要花上30分鐘至數小時，請耐心等候。

In [ ]:
import yaml
import shutil
import os
from mergekit.config import MergeConfiguration
from mergekit.merge import run_merge
from mergekit.options import MergeOptions

# 清除舊的合併模型
if os.path.exists("./merged_model"):
    shutil.rmtree("./merged_model")

# 讀取設定檔
with open("merge_config.yml", "r") as f:
    merge_config = MergeConfiguration.model_validate(yaml.safe_load(f))

# 執行合併（random_seed=42 讓 DARE 隨機剪枝結果可重現）
run_merge(
    merge_config,
    out_path="./merged_model",
    options=MergeOptions(
        cuda=True,
        lazy_unpickle=True,
        allow_crimes=True,
        random_seed=42,
    ),
)
print("合併完成！")

## 2.3 確認合併輸出

In [ ]:
import os

output_dir = "./merged_model"
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    total_size = sum(os.path.getsize(os.path.join(output_dir, f)) for f in files)
    print(f"\n✅ 合併完成！檔案數: {len(files)}, 總大小: {total_size / 1e9:.2f} GB")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f}: {size / 1e6:.1f} MB")
else:
    print("❌ 合併失敗，請檢查上方錯誤訊息")

# **Section 3: Inference — 測試合併後的模型**

## 3.1 載入合併後的模型

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

# Install sentencepiece if not already installed, as it's often needed for tokenizers.
!pip install -U sentencepiece tiktoken
output_dir = "./merged_model"

print("載入合併後的模型中...")
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForCausalLM.from_pretrained(
    output_dir, torch_dtype=torch.float16, device_map="auto"
)
print("載入完成！")

prompt = "東京の人口は"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
print(f"測試輸出: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

## 3.2 在日文數學題上測試合併模型

使用與 Section 1 相同的題目與評估方式。
Inference 時間可能會需要1-2小時，請耐心等待

In [ ]:
import json
import re

# 1. 載入題目
with open("/content/jp_math_question.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

############################################################
# 2. Prompt 模板
USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

# ⛔ 嚴禁修改 USER_PROMPT_TEMPLATE！
# 本作業聚焦於 Model Merging 技術本身，
# 而非 Prompt Engineering，因此為確保所有同學的
# 評測條件一致，無論任何情況皆請勿修改以下 prompt。
# 違反此規則將視為違規。
############################################################

# 3. 推論 + 評分
def extract_number(text):
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

correct = 0
results = []

print(f"{'='*60}")
print("📊 評估合併模型 (Merged Model)")
print(f"{'='*60}\n")

for i, item in enumerate(questions):
    prompt = USER_PROMPT_TEMPLATE.format(question=item["question"])

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            repetition_penalty=1.1
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer_part = full_output[len(prompt):].strip()
    predicted = extract_number(answer_part)
    expected = item["answer_number"]
    is_correct = predicted == expected

    if is_correct:
        correct += 1

    results.append({
        "id": i + 1,
        "correct": is_correct,
        "predicted": predicted,
        "expected": expected,
        "raw": answer_part,
    })

    mark = "✅" if is_correct else "❌"
    print(f"{mark} Q{i+1:02d} | 預測: {str(predicted):>6s} | 正解: {expected:>6s}")
    print(f"     | 題目: {item['question'][:60]}")
    print(f"     | raw: {answer_part[:120]}")
    print()

# 4. 總結
print(f"{'='*50}")
print(f"🎯 Merged Model 正確率: {correct}/{len(questions)} ({correct/len(questions)*100:.1f}%)")
print(f"{'='*50}")

# ==========================================
# 5. 儲存結果為 JSON
# ==========================================
submission_path = "/content/submission.json"

# 若你只想保留 id 和 raw
submission_data = [
    {
        "id": r["id"],
        "raw": r["raw"]
    }
    for r in results
]

with open(submission_path, "w", encoding="utf-8") as f:
    json.dump(submission_data, f, ensure_ascii=False, indent=2)

print(f"\n📁 結果已儲存至 {submission_path}")

In [ ]:
# 備存 submission.json 到 Google Drive
import shutil, os
drive_backup_dir = "/content/drive/MyDrive/ML2026/HW7"
os.makedirs(drive_backup_dir, exist_ok=True)
shutil.copy(submission_path, os.path.join(drive_backup_dir, "submission.json"))
print(f"✅ submission.json 已備存至 {drive_backup_dir}/submission.json")

# **Section 4: Grid Search — 自動試誤參數搜索（Overnight）**

自動跑 15 組 dare_ties 參數組合，每組合併 → 評估 20 題 → 存結果到 Drive。
Crash 也不會遺失中間結果（JSONL 格式，每組完成立即寫入）。

結果檔：`/content/drive/MyDrive/ML2026/HW7/grid_search_results.jsonl`

In [11]:
# ================================================================
# Section 4.1 — Grid Search Round 3：圍繞 linear 0.6/0.4 精細搜索
# ================================================================
# 上輪發現：linear_6040 = 80%（最佳），task_arithmetic_0.5 = 75%
# 本輪目標：找出 linear 甜蜜點 + 修好 slerp + task_arithmetic 精調

SHISA  = "augmxnt/shisa-gamma-7b-v1"
WIZARD = "WizardLMTeam/WizardMath-7B-V1.1"

def linear_flat(sw, ww):
    return f"""models:
  - model: {SHISA}
    parameters:
      weight:
        - value: {sw}
  - model: {WIZARD}
    parameters:
      weight:
        - value: {ww}
merge_method: linear
dtype: float16"""

def linear_grad(s_attn, s_mlp, s_def, w_attn, w_mlp, w_def):
    return f"""models:
  - model: {SHISA}
    parameters:
      weight:
        - filter: self_attn
          value: {s_attn}
        - filter: mlp
          value: {s_mlp}
        - value: {s_def}
  - model: {WIZARD}
    parameters:
      weight:
        - filter: self_attn
          value: {w_attn}
        - filter: mlp
          value: {w_mlp}
        - value: {w_def}
merge_method: linear
dtype: float16"""

def task_arith(weight):
    return f"""models:
  - model: {WIZARD}
    parameters:
      weight: {weight}
merge_method: task_arithmetic
base_model: {SHISA}
dtype: float16"""

def slerp_flat(t):
    return f"""models:
  - model: {SHISA}
  - model: {WIZARD}
merge_method: slerp
base_model: {SHISA}
parameters:
  t: {t}
dtype: float16"""

def slerp_grad(s_attn, s_mlp, s_def):
    return f"""models:
  - model: {SHISA}
  - model: {WIZARD}
merge_method: slerp
base_model: {SHISA}
parameters:
  t:
    - filter: self_attn
      value: {s_attn}
    - filter: mlp
      value: {s_mlp}
    - value: {s_def}
dtype: float16"""

SEARCH_CONFIGS = [
    # ── Linear 精調（圍繞 0.6/0.4 甜蜜點）────────────────────────
    {
        "name": "L1_linear_6040_ref",
        "desc": "[linear] shisa=0.6, wizard=0.4（80% 最佳重現）",
        "yaml": linear_flat(0.6, 0.4),
    },
    {
        "name": "L2_linear_5842",
        "desc": "[linear] shisa=0.58, wizard=0.42",
        "yaml": linear_flat(0.58, 0.42),
    },
    {
        "name": "L3_linear_5545",
        "desc": "[linear] shisa=0.55, wizard=0.45",
        "yaml": linear_flat(0.55, 0.45),
    },
    {
        "name": "L4_linear_5050",
        "desc": "[linear] shisa=0.5, wizard=0.5",
        "yaml": linear_flat(0.5, 0.5),
    },
    {
        "name": "L5_linear_6238",
        "desc": "[linear] shisa=0.62, wizard=0.38",
        "yaml": linear_flat(0.62, 0.38),
    },
    {
        "name": "L6_linear_6535",
        "desc": "[linear] shisa=0.65, wizard=0.35",
        "yaml": linear_flat(0.65, 0.35),
    },
    # ── Linear + layer gradient（甜蜜點比例 + 細緻 gradient）────────
    {
        "name": "L7_linear_grad_6040base",
        "desc": "[linear] 0.6/0.4 基礎比例 + layer gradient",
        "yaml": linear_grad(
            s_attn=[0.7, 0.6, 0.5], s_mlp=[0.7, 0.6, 0.5], s_def=0.6,
            w_attn=[0.3, 0.4, 0.5], w_mlp=[0.3, 0.4, 0.5], w_def=0.4,
        ),
    },
    {
        "name": "L8_linear_grad_steep",
        "desc": "[linear] steep gradient：早layer極偏shisa，晚layer接近5050",
        "yaml": linear_grad(
            s_attn=[0.8, 0.65, 0.5], s_mlp=[0.8, 0.6, 0.45], s_def=0.6,
            w_attn=[0.2, 0.35, 0.5], w_mlp=[0.2, 0.4, 0.55], w_def=0.4,
        ),
    },
    {
        "name": "L9_linear_grad_mlp_boost",
        "desc": "[linear] MLP 晚層大量給 wizard（attn 維持 0.6/0.4）",
        "yaml": linear_grad(
            s_attn=[0.6, 0.6, 0.6], s_mlp=[0.7, 0.6, 0.45], s_def=0.6,
            w_attn=[0.4, 0.4, 0.4], w_mlp=[0.3, 0.4, 0.55], w_def=0.4,
        ),
    },
    # ── Task Arithmetic 精調（weight=0.4 理論等於 linear_6040）──────
    {
        "name": "T1_taskarith_w04",
        "desc": "[task_arithmetic] weight=0.4（理論等於 linear_6040）",
        "yaml": task_arith(0.4),
    },
    {
        "name": "T2_taskarith_w035",
        "desc": "[task_arithmetic] weight=0.35",
        "yaml": task_arith(0.35),
    },
    {
        "name": "T3_taskarith_w045",
        "desc": "[task_arithmetic] weight=0.45",
        "yaml": task_arith(0.45),
    },
    {
        "name": "T4_taskarith_w06",
        "desc": "[task_arithmetic] weight=0.6（更多 wizard task vector）",
        "yaml": task_arith(0.6),
    },
    # ── Slerp（修正 base_model，t=0 → shisa，t=1 → wizard）─────────
    {
        "name": "S1_slerp_t03",
        "desc": "[slerp] t=0.3（球面插值，接近 linear_7030 方向）",
        "yaml": slerp_flat(0.3),
    },
    {
        "name": "S2_slerp_t04",
        "desc": "[slerp] t=0.4（接近 linear_6040 方向）",
        "yaml": slerp_flat(0.4),
    },
    {
        "name": "S3_slerp_t05",
        "desc": "[slerp] t=0.5（正中間）",
        "yaml": slerp_flat(0.5),
    },
    {
        "name": "S4_slerp_grad",
        "desc": "[slerp] layer gradient t（早layer近shisa，晚layer近wizard）",
        "yaml": slerp_grad(
            s_attn=[0.2, 0.35, 0.5],
            s_mlp=[0.2, 0.4, 0.55],
            s_def=0.35,
        ),
    },
]

print(f"✅ 共 {len(SEARCH_CONFIGS)} 組 config 定義完成：")
for c in SEARCH_CONFIGS:
    print(f"  {c['name']}: {c['desc']}")


✅ 共 21 組 config 定義完成：
  A1_daretie_d07_grad_std: [dare_ties] density=0.7, 標準gradient（70% baseline 重現）
  A2_daretie_d075_grad_std: [dare_ties] density=0.75, 標準gradient
  A3_daretie_d08_grad_std: [dare_ties] density=0.8, 標準gradient
  A4_daretie_d07_flat05: [dare_ties] density=0.7, flat weight=0.5
  A5_daretie_d07_flat07: [dare_ties] density=0.7, flat weight=0.7
  A6_daretie_d07_steep: [dare_ties] density=0.7, steep gradient（早layer極少wizard）
  A7_daretie_d08_steep: [dare_ties] density=0.8, steep gradient
  B1_ties_d07_w03: [ties] density=0.7, weight=0.3, normalize=true（確定性）
  B2_ties_d07_w05: [ties] density=0.7, weight=0.5
  B3_ties_d08_w03: [ties] density=0.8, weight=0.3
  B4_ties_d07_grad: [ties] density=0.7, 標準gradient
  C1_darelin_d07_w03: [dare_linear] density=0.7, weight=0.3（無sign consensus）
  C2_darelin_d07_grad: [dare_linear] density=0.7, 標準gradient
  C3_darelin_d07_steep: [dare_linear] density=0.7, steep gradient
  D1_linear_7030: [linear] shisa=0.7, wizard=0.3（純加權平均）
  D2_linear_

In [12]:
# ================================================================
# Section 4.2 — Grid Search 主循環（可中途 Crash 重跑，自動跳過已完成）
# ================================================================
import yaml, json, shutil, os, gc, time, datetime
import torch
from mergekit.config import MergeConfiguration
from mergekit.merge import run_merge
from mergekit.options import MergeOptions
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

RESULTS_FILE = "/content/drive/MyDrive/ML2026/HW7/grid_search_results_r3.jsonl"
MERGED_DIR   = "./merged_model"
QUESTION_FILE = "/content/jp_math_question.json"

USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

def extract_number(text):
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

with open(QUESTION_FILE, "r", encoding="utf-8") as f:
    questions = json.load(f)

done_names = set()
os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        for line in f:
            try:
                r = json.loads(line)
                done_names.add(r["name"])
            except:
                pass
    if done_names:
        print(f"⏭️  已完成，跳過: {sorted(done_names)}")

total = len(SEARCH_CONFIGS)
for idx, cfg in enumerate(SEARCH_CONFIGS, 1):
    if cfg["name"] in done_names:
        print(f"[{idx}/{total}] ⏭️  Skip {cfg['name']}")
        continue

    print(f"\n{'='*60}")
    print(f"[{idx}/{total}] 🚀 {cfg['name']}")
    print(f"       {cfg['desc']}")
    print(f"{'='*60}")
    t_start = time.time()

    with open("merge_config.yml", "w") as f:
        f.write(cfg["yaml"])

    if os.path.exists(MERGED_DIR):
        shutil.rmtree(MERGED_DIR)

    # Merge
    try:
        merge_cfg = MergeConfiguration.model_validate(yaml.safe_load(cfg["yaml"]))
        run_merge(
            merge_cfg,
            out_path=MERGED_DIR,
            options=MergeOptions(cuda=True, lazy_unpickle=True, allow_crimes=True, random_seed=42),
        )
        t_merge = time.time() - t_start
        print(f"✅ Merge: {t_merge/60:.1f} min")
    except Exception as e:
        print(f"❌ Merge failed: {e}")
        with open(RESULTS_FILE, "a") as f:
            f.write(json.dumps({"name": cfg["name"], "desc": cfg["desc"],
                                "error": str(e), "stage": "merge",
                                "timestamp": datetime.datetime.now().isoformat()},
                               ensure_ascii=False) + "\n")
        continue

    # Load model
    try:
        tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
        model = AutoModelForCausalLM.from_pretrained(
            MERGED_DIR, torch_dtype=torch.float16, device_map="auto"
        )
        t_load = time.time() - t_start - t_merge
        print(f"✅ Load:  {t_load/60:.1f} min")
    except Exception as e:
        print(f"❌ Load failed: {e}")
        with open(RESULTS_FILE, "a") as f:
            f.write(json.dumps({"name": cfg["name"], "desc": cfg["desc"],
                                "error": str(e), "stage": "load",
                                "timestamp": datetime.datetime.now().isoformat()},
                               ensure_ascii=False) + "\n")
        continue

    # Evaluate
    correct = 0
    detail = []
    eval_error = None
    for i, item in enumerate(questions):
        try:
            prompt = USER_PROMPT_TEMPLATE.format(question=item["question"])
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=512, do_sample=False, repetition_penalty=1.1
                )
            full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
            answer_part = full_output[len(prompt):].strip()
            predicted = extract_number(answer_part)
            expected = item["answer_number"]
            is_correct = predicted == expected
            if is_correct:
                correct += 1
            detail.append({"q": i+1, "correct": is_correct,
                           "predicted": predicted, "expected": expected})
            mark = "✅" if is_correct else "❌"
            print(f"  {mark} Q{i+1:02d} | pred={str(predicted):>6s} | exp={expected}")
        except Exception as e:
            eval_error = str(e)
            print(f"  ❌ Q{i+1} error: {e}")
            detail.append({"q": i+1, "correct": False, "error": str(e)})

    accuracy = correct / len(questions)
    t_total = time.time() - t_start
    print(f"\n🎯 {cfg['name']}: {correct}/{len(questions)} = {accuracy*100:.1f}%")
    print(f"⏱️  Total: {t_total/60:.1f} min")

    record = {
        "name": cfg["name"],
        "desc": cfg["desc"],
        "accuracy": accuracy,
        "correct": correct,
        "total": len(questions),
        "duration_min": round(t_total / 60, 1),
        "timestamp": datetime.datetime.now().isoformat(),
        "yaml": cfg["yaml"],
        "detail": detail,
    }
    if eval_error:
        record["eval_error"] = eval_error
    with open(RESULTS_FILE, "a") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"💾 Saved → {RESULTS_FILE}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("🗑️  Model unloaded")

print(f"\n{'='*60}")
print("🏁 Grid Search 完成！")
print(f"{'='*60}")
if os.path.exists(RESULTS_FILE):
    all_results = []
    with open(RESULTS_FILE, "r") as f:
        for line in f:
            try:
                r = json.loads(line)
                if "accuracy" in r:
                    all_results.append(r)
            except:
                pass
    all_results.sort(key=lambda x: x["accuracy"], reverse=True)
    print(f"\n📊 結果排名（共 {len(all_results)} 組）：")
    for rank, r in enumerate(all_results, 1):
        print(f"  #{rank:2d} {r['accuracy']*100:.1f}% ({r['correct']}/{r['total']}) | {r['name']}")
        print(f"       {r['desc']}")



[1/21] 🚀 A1_daretie_d07_grad_std
       [dare_ties] density=0.7, 標準gradient（70% baseline 重現）


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Warmup loader cache:   0%|          | 0/2 [00:00<?, ?it/s]

...

  ✅ Q17 | pred=  2036 | exp=2036


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  ✅ Q18 | pred=  1400 | exp=1400


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


  ✅ Q19 | pred=    11 | exp=11
  ✅ Q20 | pred=     8 | exp=8

🎯 F1_taskarith_w05: 15/20 = 75.0%
⏱️  Total: 4.4 min
💾 Saved → /content/drive/MyDrive/ML2026/HW7/grid_search_results_r2.jsonl
🗑️  Model unloaded, CUDA cache cleared

🏁 Grid Search 完成！

📊 結果排名（共 18 組）：
  # 1 80.0% (16/20) | D2_linear_6040
       [linear] shisa=0.6, wizard=0.4
  # 2 75.0% (15/20) | F1_taskarith_w05
       [task_arithmetic] weight=0.5（純task vector加法）
  # 3 65.0% (13/20) | A2_daretie_d075_grad_std
       [dare_ties] density=0.75, 標準gradient
  # 4 65.0% (13/20) | A6_daretie_d07_steep
       [dare_ties] density=0.7, steep gradient（早layer極少wizard）
  # 5 65.0% (13/20) | C3_darelin_d07_steep
       [dare_linear] density=0.7, steep gradient
  # 6 65.0% (13/20) | D3_linear_grad
       [linear] layer gradient（早layer偏shisa，晚layer偏wizard）
  # 7 60.0% (12/20) | A5_daretie_d07_flat07
       [dare_ties] density=0.7, flat weight=0.7
  # 8 60.0% (12/20) | A7_daretie_d08_steep
       [dare_ties] density=0.8, steep gradient
  # 